In [4]:
# 1. Instalación de librerías para benchmarking (silenciosa)
!pip install ydata-profiling sweetviz statsmodels -q

# 2. Importaciones principales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
import sweetviz as sv

print("✓ Entorno configurado correctamente.")

✓ Entorno configurado correctamente.


/tmp/ipykernel_5812/3750952050.py:9: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [6]:
# Carga del dataset desde el repositorio público
url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df = pd.read_csv(url)

# Establecer la variable objetivo para el modelo de regresión
target_col = "charges"

print(f"Dataset cargado con éxito. Dimensiones: {df.shape}")
df.head()

Dataset cargado con éxito. Dimensiones: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [ ]:
# Generar reporte con ydata-profiling
profile = ProfileReport(df, title="Diagnóstico ydata-profiling", explorative=True)
profile.to_file("diagnostico_ydata.html")

# Generar reporte con Sweetviz
reporte_sv = sv.analyze(df, target_feat=target_col)
reporte_sv.show_html("reporte_sweetviz.html")

print("✓ Reportes de benchmarking generados: diagnostico_ydata.html y reporte_sweetviz.html")

In [ ]:
%%writefile herramientas_eda.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

class AutoEDA:
    def __init__(self, df: pd.DataFrame, target_col: str = None):
        """
        Inicializa el pipeline de EDA automatizado para un DataFrame.
        """
        self.df = df.copy()
        self.target_col = target_col
        self.num_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        self.cat_cols = self.df.select_dtypes(include=['object', 'category']).columns.tolist()

    def _punto_1_2_overview(self):
        print("=== 1 & 2. OVERVIEW Y TIPOS DE VARIABLES ===")
        print(f"Dimensiones: {self.df.shape[0]} filas × {self.df.shape[1]} columnas")
        print(f"Uso de memoria: {self.df.memory_usage(deep=True).sum() / 1024:.2f} KB")
        print(f"Variables numéricas ({len(self.num_cols)}): {self.num_cols}")
        print(f"Variables categóricas ({len(self.cat_cols)}): {self.cat_cols}\n")

    def _punto_3_4_calidad_y_nulos(self):
        print("=== 3 & 4. CALIDAD DE DATOS Y VALORES FALTANTES ===")
        duplicados = self.df.duplicated().sum()
        print(f"Filas duplicadas exactas: {duplicados}")

        nulos = self.df.isnull().sum()
        pct_nulos = (nulos / len(self.df)) * 100
        df_nulos = pd.DataFrame({'Faltantes': nulos, 'Porcentaje (%)': pct_nulos})
        print(df_nulos[df_nulos['Faltantes'] > 0])
        if df_nulos['Faltantes'].sum() == 0:
            print("✓ No se detectaron valores nulos en el dataset.\n")

    def _punto_5_6_outliers_y_estadistica(self):
        print("=== 5 & 6. ESTADÍSTICA DESCRIPTIVA Y DETECCIÓN DE OUTLIERS (IQR) ===")
        desc = self.df[self.num_cols].describe().T[['mean', 'std', 'min', '50%', 'max']]
        desc.rename(columns={'50%': 'mediana'}, inplace=True)

        # Detección de outliers mediante IQR
        outliers_dict = {}
        for col in self.num_cols:
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            outliers = ((self.df[col] < (Q1 - 1.5 * IQR)) | (self.df[col] > (Q3 + 1.5 * IQR))).sum()
            outliers_dict[col] = outliers

        desc['outliers_IQR'] = pd.Series(outliers_dict)
        print(desc)
        print("\n")

    def _punto_7_8_target_y_sesgo(self):
        print("=== 7 & 8. DIAGNÓSTICO DEL TARGET Y ASIMETRÍA (SKEWNESS) ===")
        skewness = self.df[self.num_cols].skew()
        kurtosis = self.df[self.num_cols].kurt()
        df_shape = pd.DataFrame({'Skewness': skewness, 'Kurtosis': kurtosis})
        print(df_shape)

        for col in self.num_cols:
            if abs(skewness[col]) > 1.0:
                print(f"⚠️ ALERTA: La variable '{col}' presenta sesgo severo (|skew| = {skewness[col]:.2f} > 1.0). Se sugiere transformación logarítmica log(1+x).")
        print("\n")

    def _punto_9_10_bivariado_y_vif(self):
        print("=== 9 & 10. ANÁLISIS MULTIVARIADO Y MULTICOLINEALIDAD (VIF) ===")
        # Matriz de correlación en numéricas
        corr_matrix = self.df[self.num_cols].corr()

        # Detección de alta correlación cruzada (|r| >= 0.80)
        high_corr = []
        for i in range(len(self.num_cols)):
            for j in range(i + 1, len(self.num_cols)):
                col1, col2 = self.num_cols[i], self.num_cols[j]
                val = corr_matrix.loc[col1, col2]
                if abs(val) >= 0.80:
                    high_corr.append((col1, col2, val))

        if high_corr:
            print("⚠️ Pares de variables con alta multicolinealidad (|r| >= 0.80):")
            for c1, c2, r in high_corr:
                print(f"  - {c1} vs {c2}: r = {r:.2f}")
        else:
            print("✓ No se encontraron pares de predictores con correlación lineal directa |r| >= 0.80.")

        # Cálculo de VIF
        if len(self.num_cols) > 1:
            df_vif_data = self.df[self.num_cols].dropna()
            # Eliminar target del VIF si está presente entre las numéricas
            vif_features = [c for c in self.num_cols if c != self.target_col]
            if len(vif_features) > 1:
                vif_df = pd.DataFrame()
                vif_df["Variable"] = vif_features
                vif_df["VIF"] = [variance_inflation_factor(df_vif_data[vif_features].values, i)
                                for i in range(len(vif_features))]
                print("\nFactor de Inflación de la Varianza (VIF):")
                print(vif_df)
                for _, row in vif_df.iterrows():
                    if row['VIF'] > 5.0:
                        print(f"⚠️ ALERTA: VIF elevado en '{row['Variable']}' ({row['VIF']:.2f} > 5.0). Posible multicolinealidad.")
        print("\n")

    def plot_diagnostics(self):
        """Genera la visualización en cuadrícula para distribuciones y relaciones."""
        fig, axes = plt.subplots(len(self.num_cols), 2, figsize=(12, 4 * len(self.num_cols)))

        for i, col in enumerate(self.num_cols):
            # Histograma / KDE
            sns.histplot(self.df[col], kde=True, ax=axes[i, 0], color='skyblue')
            axes[i, 0].set_title(f'Distribución de {col}')

            # Scatter con Target o Boxplot
            if self.target_col and col != self.target_col:
                sns.regplot(data=self.df, x=col, y=self.target_col, ax=axes[i, 1], scatter_kws={'alpha':0.5}, line_kws={'color':'red'})
                axes[i, 1].set_title(f'{col} vs {self.target_col}')
            else:
                sns.boxplot(x=self.df[col], ax=axes[i, 1], color='lightgreen')
                axes[i, 1].set_title(f'Boxplot de {col}')

        plt.tight_layout()
        plt.show()

    def run_analysis(self):
        """Ejecuta el pipeline completo de los 10 puntos de EDA."""
        print("==================================================")
        print("   DIAGNÓSTICO AUTOMATIZADO DE EDA (REGRESIÓN)   ")
        print("==================================================\n")
        self._punto_1_2_overview()
        self._punto_3_4_calidad_y_nulos()
        self._punto_5_6_outliers_y_estadistica()
        self._punto_7_8_target_y_sesgo()
        self._punto_9_10_bivariado_y_vif()
        self.plot_diagnostics()

In [ ]:
# Importar la clase desde el script generado
from herramientas_eda import AutoEDA

# Instanciar y correr el diagnóstico automatizado
eda = AutoEDA(df, target_col="charges")
eda.run_analysis()